# Neo4j Talk Demo: From PDF to Knowledge Graph

One notebook, run top to bottom:

1. Get a free Neo4j Aura instance
2. Connect to it from Python
3. Load the provided sample PDF (a fictional company profile written as real prose, not placeholder text)
4. Extract entities & relationships from it with simple rule-based regex (no NLP/LLM dependency)
5. Push the extracted data into Neo4j via Cypher
6. Query the graph — a relationship lookup, a path-finding query, and a "person 360" query

Only `extraction.py` (the regex extraction logic) is imported from outside this notebook. Everything else — the connection, the push, the queries — is inline below.

## 1. Get a free Neo4j Aura instance

If you don't already have one:

1. Go to https://neo4j.com/cloud/aura-free/ and sign up (or log in) — the Free tier needs no credit card.
2. Click **New Instance** → choose **AuraDB Free**.
3. Give it a name (e.g. `pu-demo`) and click **Create**.
4. Aura shows the generated password **once** — click **Download** to save the `.txt` credentials file, or copy the URI/username/password immediately. You cannot retrieve the password again later (you'd have to reset it).
5. Wait ~1-2 minutes for the instance status to turn from *Creating* to *Running*.
6. Copy `.env.example` in this folder to `.env` and fill in the three values from step 4:
   ```
   NEO4J_URI=neo4j+s://xxxxxxxx.databases.neo4j.io
   NEO4J_USERNAME=neo4j
   NEO4J_PASSWORD=<the generated password>
   ```

`.env` is only read locally by the next cell — it's never printed or committed anywhere by this notebook.

## 2. Connect to Neo4j

In [1]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase, TrustCustomCAs, TrustSystemCAs

load_dotenv()

NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# This machine sits behind a TLS-inspecting corporate proxy (Netskope) that
# re-signs outbound TLS with its own CA, which the driver's default trust
# store doesn't recognize. Point it at the combined CA bundle already set up
# for this Windows user; falls back to the system trust store elsewhere.
#
# The neo4j+s:// scheme bakes in "encrypted, system trust store only, no
# overrides" — passing trusted_certificates alongside it is rejected. To use
# a custom CA bundle we drop to the bare neo4j:// scheme and set encryption
# explicitly instead.
_netskope_bundle = os.path.expanduser(r"~\certs\combined-ca-bundle.pem")
trusted_certificates = (
    TrustCustomCAs(_netskope_bundle) if os.path.exists(_netskope_bundle) else TrustSystemCAs()
)
unencrypted_scheme_uri = NEO4J_URI.replace("neo4j+s://", "neo4j://").replace("neo4j+ssc://", "neo4j://")

driver = GraphDatabase.driver(
    unencrypted_scheme_uri,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    encrypted=True,
    trusted_certificates=trusted_certificates,
)
driver.verify_connectivity()
print("Connected to Neo4j Aura.")

Connected to Neo4j Aura.


## 3. Sample PDF

`sample_company_profile.pdf` (next to this notebook) is provided for you — a profile of a fictional company, **Nexora Systems**, written as real sentences (not lorem ipsum) so the extraction step in Section 4 has something meaningful to parse.

In [2]:
from pypdf import PdfReader

PDF_PATH = "sample_company_profile.pdf"

reader = PdfReader(PDF_PATH)
raw_text = "\n".join(page.extract_text() for page in reader.pages)

print(raw_text[:500], "...")

Nexora Systems is a mid-sized technology company that builds data infrastructure and analytics
tools for enterprise clients. Nexora Systems is headquartered in Austin, Texas.
The company is organized into four departments: Data Engineering, Cloud Platform, Research, and
Customer Success.
Aisha Khan is the Head of Data Engineering at Nexora Systems. She joined Nexora Systems in
2019 after leading data teams at two earlier startups. Marcus Chen is the Head of Cloud Platform at
Nexora Systems. Priy ...


## 4. Extract entities & relationships (rule-based, into CSV-ready tables)

`extraction.py` runs a fixed set of regex patterns over the text — no NLP model, no LLM call — matching sentence shapes like *"X is the Head of Y at Z."* or *"The X project is led by Y."*. It returns two tables ready to write out as CSV.

In [3]:
from extraction import build_entities_and_relationships

entities_df, relationships_df = build_entities_and_relationships(raw_text)

entities_df.to_csv("entities.csv", index=False)
relationships_df.to_csv("relationships.csv", index=False)

print(f"{len(entities_df)} entities, {len(relationships_df)} relationships")
entities_df.head(10)

24 entities, 35 relationships


,name,type
0,Aisha Khan,Person
1,Aurora,Project
2,"Austin, Texas",Location
3,Beacon,Project
4,"Berlin, Germany",Location
5,Cascade,Project
6,Cloud Platform,Department
7,Customer Success,Department
8,Daniel Osei,Person
9,Data Engineering,Department


In [4]:
relationships_df.head(10)

,source,source_type,rel_type,target,target_type
0,Aisha Khan,Person,IS_HEAD_OF,Data Engineering,Department
1,Data Engineering,Department,PART_OF,Nexora Systems,Company
2,Marcus Chen,Person,IS_HEAD_OF,Cloud Platform,Department
3,Cloud Platform,Department,PART_OF,Nexora Systems,Company
4,Priya Nair,Person,IS_HEAD_OF,Customer Success,Department
5,Customer Success,Department,PART_OF,Nexora Systems,Company
6,Daniel Osei,Person,IS_HEAD_OF,Research,Department
7,Research,Department,PART_OF,Nexora Systems,Company
8,Liam Foster,Person,WORKS_IN,Data Engineering,Department
9,Sofia Ramirez,Person,WORKS_IN,Cloud Platform,Department


## 5. Push to Neo4j (Cypher)

One `MERGE` per entity type, one `MERGE` per (source type, relationship type, target type) combination — `MERGE` makes this safe to re-run without creating duplicates. Node labels and relationship types come from the fixed vocabulary produced in Section 4, not free text, so they're safe to interpolate directly into the Cypher strings below.

In [5]:
VALID_LABELS = {"Person", "Company", "Department", "Project", "Location", "Client"}
VALID_REL_TYPES = {
    "IS_HEAD_OF", "WORKS_IN", "PART_OF", "HEADQUARTERED_IN",
    "BASED_IN", "LEADS", "WORKS_ON", "PARTNERS_WITH", "FOR_CLIENT",
}

with driver.session() as session:
    # One node per (name, type) row, grouped by label for a single query per type.
    for entity_type, group in entities_df.groupby("type"):
        if entity_type not in VALID_LABELS:
            raise ValueError(f"Unexpected entity type: {entity_type}")
        session.run(
            f"UNWIND $names AS name MERGE (n:{entity_type} {{name: name}})",
            names=group["name"].tolist(),
        )

    # One relationship per row, grouped by (source_type, rel_type, target_type).
    group_cols = ["source_type", "rel_type", "target_type"]
    for (source_type, rel_type, target_type), group in relationships_df.groupby(group_cols):
        if source_type not in VALID_LABELS or target_type not in VALID_LABELS:
            raise ValueError(f"Unexpected node type in relationship: {source_type}/{target_type}")
        if rel_type not in VALID_REL_TYPES:
            raise ValueError(f"Unexpected relationship type: {rel_type}")
        session.run(
            f"""
            UNWIND $rows AS row
            MATCH (a:{source_type} {{name: row.source}})
            MATCH (b:{target_type} {{name: row.target}})
            MERGE (a)-[:{rel_type}]->(b)
            """,
            rows=group[["source", "target"]].to_dict("records"),
        )

print("Pushed to Neo4j.")

Pushed to Neo4j.


## 6. Query it

### 6a. Relationship lookup

In [6]:
import pandas as pd


def run_query(driver, cypher, **params):
    """Run a Cypher query and return the results as a DataFrame."""
    with driver.session() as session:
        result = session.run(cypher, **params)
        return pd.DataFrame([record.data() for record in result])

In [7]:
who_works_on_aurora = run_query(
    driver,
    """
    MATCH (p:Person)-[:WORKS_ON]->(proj:Project {name: $project})
    RETURN p.name AS person, proj.name AS project
    """,
    project="Aurora",
)
who_works_on_aurora

,person,project
0,Daniel Osei,Aurora
1,Liam Foster,Aurora


In [8]:
departments_by_location = run_query(
    driver,
    """
    MATCH (d:Department)-[:PART_OF]->(:Company {name: $company})
    MATCH (d)-[:BASED_IN]->(l:Location)
    RETURN d.name AS department, l.name AS location
    ORDER BY department
    """,
    company="Nexora Systems",
)
departments_by_location

,department,location
0,Cloud Platform,"Denver, Colorado"
1,Customer Success,"Austin, Texas"
2,Data Engineering,"Austin, Texas"
3,Research,"Berlin, Germany"


### 6b. Path finding

This is the query that's awkward in SQL and natural in Cypher: the shortest path between two people who never appear in the same sentence — `shortestPath` traverses through whatever departments/projects/clients connect them.

Note: arrows below are shown for readability only; `-[*..6]-` is an *undirected* traversal, so a relationship's arrow direction here doesn't necessarily match the direction it was stored in.

In [9]:
PERSON_A = "Liam Foster"
PERSON_B = "Ethan Wallace"

with driver.session() as session:
    result = session.run(
        """
        MATCH path = shortestPath((a:Person {name: $person_a})-[*..6]-(b:Person {name: $person_b}))
        RETURN path
        """,
        person_a=PERSON_A,
        person_b=PERSON_B,
    )
    record = result.single()

if record is None:
    print(f"No path found between {PERSON_A} and {PERSON_B}")
else:
    path = record["path"]
    node_labels = [f"{next(iter(node.labels))}:{node['name']}" for node in path.nodes]
    rel_types = [rel.type for rel in path.relationships]

    formatted = node_labels[0]
    for rel_type, node_label in zip(rel_types, node_labels[1:]):
        formatted += f" --[{rel_type}]--> {node_label}"

    print(f"Path length: {len(rel_types)} hops")
    print(formatted)

Path length: 4 hops
Person:Liam Foster --[WORKS_IN]--> Department:Data Engineering --[PART_OF]--> Company:Nexora Systems --[PART_OF]--> Department:Research --[WORKS_IN]--> Person:Ethan Wallace


### 6c. Customer / Person 360

The "customer 360" pitch: everything connected to one person within 2 hops — their department, their company, their colleagues, the projects they lead or work on, and the clients on those projects — in a single query. No JOINs, no schema-specific query per relationship type, just a hop count.

In [10]:
PERSON_NAME = "Aisha Khan"

person_360 = run_query(
    driver,
    """
    MATCH (p:Person {name: $name})-[*1..2]-(connected)
    WHERE connected <> p
    RETURN DISTINCT labels(connected)[0] AS type, connected.name AS name
    ORDER BY type, name
    """,
    name=PERSON_NAME,
)
person_360

,type,name
0,Client,Fortis Bank
1,Company,Nexora Systems
2,Department,Data Engineering
3,Location,"Austin, Texas"
4,Person,Daniel Osei
5,Person,Liam Foster
6,Person,Marcus Chen
7,Person,Sofia Ramirez
8,Project,Aurora
9,Project,Beacon


In [11]:
driver.close()